In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
import torch

from typing import List, Dict, Any, Tuple, Iterable, Optional
from sklearn.model_selection import train_test_split
import copy

from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# Data

In [ ]:
df = pd.read_parquet("hf://datasets/OrdalieTech/Ordalie-FR-STS-benchmark/data/test-00000-of-00001.parquet")
df.head(3)

In [ ]:
df.value_counts("score")

# Undersampling

In [ ]:
def undersample_binary_from_dataframe(df: pd.DataFrame, column_name: str="score", random_state: int=42):
    """
    Undersamples a binary dataframe based on a specified column.

    Args:
        df (pd.DataFrame): The input dataframe.
        column_name (str, optional): The column name to use for undersampling. Defaults to "score".
        random_state (int, optional): The random state for reproducibility. Defaults to 42.

    Returns:
        pd.DataFrame: The undersampled dataframe.
    """

    low_category = df.value_counts(column_name).sort_values().index[0]
    high_category = df.value_counts(column_name).sort_values().index[1]

    lower_value= df.value_counts(column_name).sort_values().values[0]

    df_result = pd.concat([df[df[column_name] == low_category], df[df[column_name] == high_category].sample(lower_value, random_state=random_state)])

    return df_result

In [ ]:
df = undersample_binary_from_dataframe(df, "score")

In [ ]:
df.value_counts("score")

# Train Test Split

In [ ]:
def get_train_test_val_split(df: pd.DataFrame, test_size: float=0.2, val_size: float=0.1, random_state: int=42):
    """
    Split a DataFrame into training, testing, and validation sets.

    Parameters:
    - df (pandas.DataFrame): DataFrame to split.
    - test_size (float): Size of the testing set.
    - val_size (float): Size of the validation set.
    - random_state (int): Random state for reproducibility.

    Returns:
    - df_train (pandas.DataFrame): Training set.
    - df_test (pandas.DataFrame): Testing set.
    - df_val (pandas.DataFrame): Validation set.
    """

    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)
    df_train, df_val = train_test_split(df_train, test_size=val_size, random_state=random_state)

    return df_train, df_test, df_val

In [ ]:
df_train, df_test, df_val = get_train_test_val_split(df, test_size=0.2, val_size=0.1, random_state=42)

In [ ]:
def get_examples(df: pd.DataFrame, sentence1_colname: str="sentence1", sentence2_colname: str="sentence2", label_colname :str="score"):
    """
    Generate a list of InputExample objects from a DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.
        sentence1_colname (str, optional): The column name for the first sentence. Defaults to "sentence1".
        sentence2_colname (str, optional): The column name for the second sentence. Defaults to "sentence2".
        label_colname (str, optional): The column name for the label. Defaults to "score".

    Returns:
        list: A list of InputExample objects.
    """
    return [InputExample(texts=[row[sentence1_colname], row[sentence2_colname]], label=row[label_colname]) for i, row in df.iterrows()]

In [ ]:
train_examples = get_examples(df_train)
test_examples = get_examples(df_test)
val_examples = get_examples(df_val)

In [ ]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
test_dataloader = DataLoader(test_examples, shuffle=True, batch_size=16)
val_dataloader = DataLoader(val_examples, shuffle=True, batch_size=16)

# Finetune

In [ ]:
model_without_finetuning = SentenceTransformer("paraphrase-MiniLM-L6-v2")

In [ ]:
def finetune_model(model: SentenceTransformer, train_dataloader: DataLoader, val_dataloader: DataLoader, num_epochs: int=4, loss_class: Any=losses.CosineSimilarityLoss, **fit_params):
    """
    Finetune a SentenceTransformer model.

    Parameters:
    - model (SentenceTransformer): SentenceTransformer model to finetune.
    - train_dataloader (DataLoader): DataLoader containing the training data.
    - val_dataloader (DataLoader): DataLoader containing the validation data.
    - epochs (int): Number of epochs to train the model.
    - loss (Any): Loss function to use for training.

    Returns:
    - model (SentenceTransformer): Finetuned model.
    """
    to_finetune = copy.deepcopy(model)
    loss_class = losses.CosineSimilarityLoss

    loss_tu_use = loss_class(model=to_finetune)

    dev_evaluator = evaluation.EmbeddingSimilarityEvaluator.from_input_examples(val_dataloader.dataset, name='dev-eval')

    device = torch.device("mps") if torch.has_mps else torch.device("cpu")
    to_finetune.to(device)

    to_finetune.fit(
        train_objectives=[(train_dataloader, loss_tu_use)],
        evaluator=dev_evaluator,
        epochs=num_epochs,
        **fit_params
    )

    return to_finetune

In [ ]:
model_with_finetuning = finetune_model(model_without_finetuning, train_dataloader, val_dataloader, num_epochs=4, loss_class=losses.CosineSimilarityLoss, warmup_steps=100, evaluation_steps=1000)

# Test

In [ ]:
def compute_accuracy_similarity(model: SentenceTransformer, test_dataloader: DataLoader):
    """
    Computes the accuracy of a sentence similarity model on a given test dataset.

    Args:
        model (SentenceTransformer): The sentence similarity model.
        test_dataloader (DataLoader): The test dataloader containing the dataset.

    Returns:
        float: The accuracy score of the model on the test dataset.
    """
    y_true = [input_element.label for input_element in test_dataloader.dataset]

    sentence1_embs = model.encode([input_element.texts[0] for input_element in test_dataloader.dataset])
    sentence2_embs = model.encode([input_element.texts[1] for input_element in test_dataloader.dataset])

    cos_sim = cosine_similarity(sentence1_embs, sentence2_embs)
    diag = np.diagonal(cos_sim)
    y_pred = np.where(diag > 0.5, 1., 0.)

    acc_score = accuracy_score(y_true, y_pred)

    return acc_score

In [ ]:
compute_accuracy_similarity(model_without_finetuning, test_dataloader), compute_accuracy_similarity(model_with_finetuning, test_dataloader)